In [ ]:
import pandas as pd

ref_sci_gen = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/sci_gen_human.csv")
ref_cmv = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/cmv_human.csv")
ref_eli = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/eli5_human.csv")
ref_roct = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/roct_human.csv")
ref_tldr = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/tldr_human.csv")
ref_wp = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/wp_human.csv")
ref_xsum = pd.read_csv("../plagiarism_detection_dataset/deepfaketextdetect/processed/xsum_human.csv")

import numpy as np

train_all = [ref_sci_gen, ref_cmv, ref_eli,ref_roct, ref_tldr, ref_wp, ref_xsum]

for df in train_all:
    print(np.mean([len(i.split(" ")) for i in df['text']]))

In [ ]:
# random_sample = df.sample(n=3)
random_seed = 400
short = [ref_sci_gen.sample(n=500,random_state=random_seed), ref_roct.sample(n=500, random_state=random_seed), ref_tldr.sample(n=500, random_state=random_seed)]
long = [ref_cmv.sample(n=500, random_state=random_seed), ref_eli.sample(n=500, random_state=random_seed), ref_wp.sample(n=500, random_state=random_seed), ref_xsum.sample(n=500, random_state=random_seed)]

In [ ]:
short_genre = ['ref_sci_gen'] * 500 + ['ref_roct'] * 500 + ['ref_tldr'] * 500 

# Concatenate the DataFrames horizontally
short_df = pd.concat(short, axis=0).reset_index(drop=True)
#print(len(short_df))
short_df['genre']=short_genre

short_df.head(10)

In [ ]:
# random_sample = df.sample(n=3)
random_seed = 4
short2 = [ref_sci_gen.sample(n=2500,random_state=random_seed), ref_roct.sample(n=2500, random_state=random_seed), ref_tldr.sample(n=2500, random_state=random_seed)]
#long = [ref_cmv.sample(n=500, random_state=random_seed), ref_eli.sample(n=500, random_state=random_seed), ref_wp.sample(n=500, random_state=random_seed), ref_xsum.sample(n=500, random_state=random_seed)]

In [ ]:
short_genre2 = ['ref_sci_gen'] * 2500 + ['ref_roct'] * 2500 + ['ref_tldr'] * 2500 

# Concatenate the DataFrames horizontally
short_df2 = pd.concat(short2, axis=0).reset_index(drop=True)
#print(len(short_df))
short_df2['genre']=short_genre2

short_df2.head(10)

In [ ]:
short_df2 = short_df2[~short_df2['text'].isin(short_df['text'])].reset_index(drop=True)

In [ ]:
final = pd.concat([short_df,short_df2], axis=0)
print(len(final))

In [ ]:
final.to_csv("seed_doc_collection.csv", index=False)

### No plagiarism Case Generation

In [ ]:
!pip install keybert

In [ ]:
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
keyword_model = KeyBERT(model=sentence_model)

keywords = keyword_model.extract_keywords("hello world how are you", keyphrase_ngram_range=(1, 3), stop_words='english',
                              use_maxsum=True, nr_candidates=20, top_n=5)

In [ ]:
import pandas as pd
import nltk
import requests
import json
import random
import os

random.seed(20000)

API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
headers = {"Authorization": YOUR API,
        "Content-Type": "application/json",}

import openai

api_key = YOUR API # API key
openai.api_key = api_key

def get_least_sim_doc(target_doc):
    query_embedding = model.encode(target_doc, convert_to_tensor=True)

    search_hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=len(train_all))[0]
    
    return train_all['text'][int(search_hits[-1]['corpus_id'])], target_doc


def gen_chatgpt(sentences, keywords):
     prompt = f'''Based on the provided keywords and text passage, write its continuation in an academic writing style. When generating, make sure that the continuation is relevant to the keywords and flows well.
    
keywords: {keywords}
text passage: {sentences}
continuation: '''

    LLM_genrated_text = openai.ChatCompletion.create(
            model="gpt-3.5-turbo-1106", 
            max_tokens=50,
            #temperature=0.7,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']


def gen_llama2(sentences, keywords):
    prompt = f'''Based on the provided keywords and text passage, write its continuation in an academic writing style. When generating, make sure that the continuation is relevant to the keywords and flows well.
    
keywords: {keywords}
text passage: {sentences}
continuation: '''
    
    json_body = {
        "inputs": prompt,
                "parameters": {"max_new_tokens":350}
        }
    data = json.dumps(json_body)
    response = requests.request("POST", API_URL, headers=headers, data=data)
    try:
        return json.loads(response.content.decode("utf-8")), prompt
    except:
        return response
   

sci_gen = short_df[short_df['genre']=="ref_sci_gen"].reset_index(drop=True)
print(len(sci_gen))
para_results_llama2 = [] 
index = 0
for text in sci_gen['text']:
    if index >= 0:
        keywords = keyword_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english',
                              use_maxsum=True, nr_candidates=20, top_n=5)
        generation, prompt = gen_llama2(" ".join(nltk.sent_tokenize(text)[:2]), ", ".join([i[0] for i in keywords]))
        if index%10==0:
            print("index:", index)

        #para_results_llama2.append({"source_doc": text, "susp_doc": paraphrased[0]['generated_text'], "label": "yes", "obfuscation_type": "paraphrase"})

        # write files
        loc = '.'
        filename = 'llama2_no_plagiarism_ref_sci_gen.json'

        # make dir
        os.makedirs(f"{loc}", exist_ok=True)

        try:
            with open(f'{loc}/{filename}', 'r') as file:
                existing_data = json.load(file)
        except FileNotFoundError:
            # If the file doesn't exist, create an empty list as the starting point
            existing_data = []


        # Step 3: Modify the data structure by adding the new item
        new_item = {"source_doc": text, "susp_doc": generation[0]['generated_text'].replace(prompt,""), "label": "no", "prefix": " ".join(nltk.sent_tokenize(text)[:2]), "keywords": ", ".join([i[0] for i in keywords])} # Replace with your item
        existing_data.append(new_item)

        # Step 4: Write the updated data structure back to the JSON file
        with open(f'{loc}/{filename}', 'w') as file:
            json.dump(existing_data, file, indent=4)
        
    index+=1

###  paraphrase generation

In [ ]:
import pandas as pd
import nltk
import requests
import json
import random
import os

random.seed(1000)

API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
headers = {"Authorization": YOUR API,
        "Content-Type": "application/json",}

import openai

api_key = YOUR API # API key
openai.api_key = api_key

def get_least_sim_doc(target_doc):
    query_embedding = model.encode(target_doc, convert_to_tensor=True)

    search_hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=len(train_all))[0]
    
    return train_all['text'][int(search_hits[-1]['corpus_id'])], target_doc


def gen_chatgpt(sentences):
    prompt = f'''Rephrase the following text while keeping its meaning.
    
text: New cryptographic techniques such as homomorphic encryption (HE) allow computations to be outsourced to and evaluated blindfolded in a resourceful cloud. These computations often require private data owned by multiple participants, engaging in joint evaluation of some functions. For example, Genome-Wide Association Study (GWAS) is becoming feasible because of recent proliferation of genome sequencing technology. Due to the sensitivity of genomic data, these data should be encrypted using different keys. 
paraphrased: Novel cryptographic methods, such homomorphic encryption (HE), enable computations to be delegated to and assessed in a creative cloud while wearing blinders. These calculations frequently call for the confidential information of several participants in order to evaluate specific functions jointly. For instance, the rapid expansion of genome sequencing technology has made the Genome-Wide Association Study (GWAS) viable. Genomic data is sensitive, hence it is best to encrypt it with distinct keys. 

text: Susie was about to get her first kiss. She liked her friend Fred a lot. Fred leaned in and kissed her. But Susie did not like it. It was gross to her. 
paraphrased: Susie's first kiss was just around the corner. She was very fond of her friend Fred. Fred kissed her, leaning in. Susie, though, did not enjoy it. She thought it was disgusting.

text: Python 3.9 was released on October 5 and as with every new version of Python, it includes new, improved, and deprecated features. A link to the complete list of changes is provided at the beginning of the article. This article is a tutorial that covers some of Python 3.9's new features, including proper time zone support, simple updating of dictionaries, more flexible decorators, a more powerful Python parser, and more.
paraphrased: Like all major releases of Python, Python 3.9 was made available on October 5 and comes with both deprecated and new features. The article's opening contains a link to the full list of modifications. This course covers some of the new features of Python 3.9, such as correct time zone support, easy dictionary updates, more versatile decorators, a more potent Python parser, and more.

text: {sentences}
paraphrased: '''

    LLM_genrated_text = openai.ChatCompletion.create(
            model = 'gpt-3.5-turbo-1106',
            messages=[
                {"role": "user", "content": prompt},
              ],
    )

    return LLM_genrated_text['choices'][0]['message']['content']



def gen_llama2(sentences):
    prompt = f'''Rephrase the following text while keeping its meaning.
    
text: New cryptographic techniques such as homomorphic encryption (HE) allow computations to be outsourced to and evaluated blindfolded in a resourceful cloud. These computations often require private data owned by multiple participants, engaging in joint evaluation of some functions. For example, Genome-Wide Association Study (GWAS) is becoming feasible because of recent proliferation of genome sequencing technology. Due to the sensitivity of genomic data, these data should be encrypted using different keys. 
paraphrased: Novel cryptographic methods, such homomorphic encryption (HE), enable computations to be delegated to and assessed in a creative cloud while wearing blinders. These calculations frequently call for the confidential information of several participants in order to evaluate specific functions jointly. For instance, the rapid expansion of genome sequencing technology has made the Genome-Wide Association Study (GWAS) viable. Genomic data is sensitive, hence it is best to encrypt it with distinct keys. 

text: Susie was about to get her first kiss. She liked her friend Fred a lot. Fred leaned in and kissed her. But Susie did not like it. It was gross to her. 
paraphrased: Susie's first kiss was just around the corner. She was very fond of her friend Fred. Fred kissed her, leaning in. Susie, though, did not enjoy it. She thought it was disgusting.

text: Python 3.9 was released on October 5 and as with every new version of Python, it includes new, improved, and deprecated features. A link to the complete list of changes is provided at the beginning of the article. This article is a tutorial that covers some of Python 3.9's new features, including proper time zone support, simple updating of dictionaries, more flexible decorators, a more powerful Python parser, and more.
paraphrased: Like all major releases of Python, Python 3.9 was made available on October 5 and comes with both deprecated and new features. The article's opening contains a link to the full list of modifications. This course covers some of the new features of Python 3.9, such as correct time zone support, easy dictionary updates, more versatile decorators, a more potent Python parser, and more.

text: {sentences}
paraphrased: '''
    
    json_body = {
        "inputs": prompt,
                "parameters": {"max_new_tokens":100}
        }
    data = json.dumps(json_body)
    response = requests.request("POST", API_URL, headers=headers, data=data)
    try:
        return prompt, json.loads(response.content.decode("utf-8"))
    except:
        return response
   

sci_gen = short_df[short_df['genre']=="ref_tldr"].reset_index(drop=True)
print(len(sci_gen))
para_results_llama2 = [] 
index = 0
for text in sci_gen['text']:
  
    paraphrased = gen_chatgpt(text)

    if index%10==0:
        print("index:", index)
        
       

    # write files
    loc = '.'
    filename = 'ref_tldr_paraphrase_gpt3_remaining.json'

    # make dir
    os.makedirs(f"{loc}", exist_ok=True)

    try:
        with open(f'{loc}/{filename}', 'r') as file:
            existing_data = json.load(file)
    except FileNotFoundError:
        # If the file doesn't exist, create an empty list as the starting point
        existing_data = []


    # Step 3: Modify the data structure by adding the new item
    new_item = {"source_doc": text, "susp_doc": paraphrased, "label": "yes", "obfuscation_type": "paraphrase"} # Replace with your item
    existing_data.append(new_item)

    # Step 4: Write the updated data structure back to the JSON file
    with open(f'{loc}/{filename}', 'w') as file:
        json.dump(existing_data, file, indent=4)
     
    index+=1

###  summary generation

In [ ]:
import pandas as pd
import nltk
import requests
import json
import random
import os

random.seed(20000)

API_URL = "https://api-inference.huggingface.co/models/meta-llama/Llama-2-70b-chat-hf"
headers = {"Authorization":  YOUR API,
        "Content-Type": "application/json",}

import openai

api_key = YOUR API # API key
openai.api_key = api_key

def get_least_sim_doc(target_doc):
    query_embedding = model.encode(target_doc, convert_to_tensor=True)

    search_hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=len(train_all))[0]
    
    return train_all['text'][int(search_hits[-1]['corpus_id'])], target_doc


def gen_chatgpt(sentences):
    prompt = f'''Summarize the following text in 1-3 sentences.
    
text: New cryptographic techniques such as homomorphic encryption (HE) allow computations to be outsourced to and evaluated blindfolded in a resourceful cloud. These computations often require private data owned by multiple participants, engaging in joint evaluation of some functions. For example, Genome-Wide Association Study (GWAS) is becoming feasible because of recent proliferation of genome sequencing technology. Due to the sensitivity of genomic data, these data should be encrypted using different keys. 
summarized: Homomorphic encryption (HE) enables computations to be outsourced and evaluated in a cloud, requiring private data from multiple participants, enabling Genome-Wide Association Study (GWAS) due to sensitivity.

text: Susie was about to get her first kiss. She liked her friend Fred a lot. Fred leaned in and kissed her. But Susie did not like it. It was gross to her. 
summarized: Susie, fond of her friend Fred, was about to receive her first kiss, but she found it gross and disliked the gesture.

text: Python 3.9 was released on October 5 and as with every new version of Python, it includes new, improved, and deprecated features. A link to the complete list of changes is provided at the beginning of the article. This article is a tutorial that covers some of Python 3.9's new features, including proper time zone support, simple updating of dictionaries, more flexible decorators, a more powerful Python parser, and more.
summarized: Python 3.9, released on October 5, introduced new features such as time zone support, dictionary updating, flexible decorators, and a powerful Python parser, along with a link to the complete list of changes.

text: {sentences}
summarized: '''

    LLM_genrated_text = openai.ChatCompletion.create(
            model="gpt-3.5-turbo-1106", #"text-davinci-003", #"gpt-3.5-turbo-1106" #gpt-4-1106-preview
            max_tokens=50,
            messages=[
                {"role": "user", "content": prompt},
              ],
    )
    return LLM_genrated_text['choices'][0]['message']['content']



def gen_llama2(sentences):
    prompt = f'''Summarize the following text in 1-3 sentences.
    
text: New cryptographic techniques such as homomorphic encryption (HE) allow computations to be outsourced to and evaluated blindfolded in a resourceful cloud. These computations often require private data owned by multiple participants, engaging in joint evaluation of some functions. For example, Genome-Wide Association Study (GWAS) is becoming feasible because of recent proliferation of genome sequencing technology. Due to the sensitivity of genomic data, these data should be encrypted using different keys. 
summarized: Homomorphic encryption (HE) enables computations to be outsourced and evaluated in a cloud, requiring private data from multiple participants, enabling Genome-Wide Association Study (GWAS) due to sensitivity.

text: Susie was about to get her first kiss. She liked her friend Fred a lot. Fred leaned in and kissed her. But Susie did not like it. It was gross to her. 
summarized: Susie, fond of her friend Fred, was about to receive her first kiss, but she found it gross and disliked the gesture.

text: Python 3.9 was released on October 5 and as with every new version of Python, it includes new, improved, and deprecated features. A link to the complete list of changes is provided at the beginning of the article. This article is a tutorial that covers some of Python 3.9's new features, including proper time zone support, simple updating of dictionaries, more flexible decorators, a more powerful Python parser, and more.
summarized: Python 3.9, released on October 5, introduced new features such as time zone support, dictionary updating, flexible decorators, and a powerful Python parser, along with a link to the complete list of changes.

text: {sentences}
summarized: '''
    
    json_body = {
        "inputs": prompt,
                "parameters": {"max_new_tokens":50}
        }
    data = json.dumps(json_body)
    response = requests.request("POST", API_URL, headers=headers, data=data)
    try:
        return json.loads(response.content.decode("utf-8"))
    except:
        return response
   

sci_gen = short_df2[short_df2['genre']=="ref_tldr"].reset_index(drop=True)
print(len(sci_gen))
para_results_llama2 = [] 
index = 0
for text in sci_gen['text']:
    if index >= 0:
        summary = gen_llama2(text)
        #print(paraphrased[0]['generated_text'])
        if index%10==0:
            print("index:", index)

        #para_results_llama2.append({"source_doc": text, "susp_doc": paraphrased[0]['generated_text'], "label": "yes", "obfuscation_type": "paraphrase"})


        # write files
        loc = '.'
        filename = 'llama2_summary_ref_tldr_v2.json'

        # make dir
        os.makedirs(f"{loc}", exist_ok=True)

        try:
            with open(f'{loc}/{filename}', 'r') as file:
                existing_data = json.load(file)
        except FileNotFoundError:
            # If the file doesn't exist, create an empty list as the starting point
            existing_data = []


        # Step 3: Modify the data structure by adding the new item
        new_item = {"source_doc": text, "susp_doc": summary[0]['generated_text'], "label": "yes", "obfuscation_type": "summary"} # Replace with your item
        existing_data.append(new_item)

        # Step 4: Write the updated data structure back to the JSON file
        with open(f'{loc}/{filename}', 'w') as file:
            json.dump(existing_data, file, indent=4)
        
    index+=1